## Milestone 4 — Learning Curve Diagnoses & Case D Experiment

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Case Summary Table:
# Case A: Train 62%, Val 60% (Flat) -> High Bias / Underfitting
# Case B: Train 99%, Val 78% (Val rising) -> High Variance / Overfitting
# Case C: Train 85%, Val 91% (Inverted) -> Training noise (Dropout) active in train(), off in eval()
# Case D: Train 88%, Val 70% (Gap stubborn with 4x data) -> Covariate shift or noisy labels


**The Case D Experiment: Distinguishing Covariate Shift from Label Noise**

In [4]:
# In Case D, 4x data didn't close the 18% gap.
# Two explanations:
# 1. Covariate Shift: Train and Val have different feature distributions P(X).
# 2. Label Noise: Val ground-truth labels are noisy, putting an artificial ceiling on accuracy.

# Step 1: Adversarial Validation
# We strip labels y, label Train as z=0 and Val as z=1, and train a classifier to separate them.
# If AUC >> 0.50 -> Feature distributions differ (Covariate Shift confirmed).
# If AUC ~ 0.50 -> Feature distributions are identical (Label Noise confirmed).

# Simulation 1: True Covariate Shift (Train from N(0, 1), Val shifted by +1.5)
X_tr_shift = np.random.randn(2000, 10)
X_val_shift = np.random.randn(500, 10) + 1.5

# Simulation 2: Identical distributions with 25% noisy validation labels
X_tr_noise = np.random.randn(2000, 10)
X_val_noise = np.random.randn(500, 10)

def adversarial_auc(X_tr, X_val):
    X_comb = np.vstack([X_tr, X_val])
    z = np.concatenate([np.zeros(len(X_tr)), np.ones(len(X_val))])
    X_a_tr, X_a_te, z_a_tr, z_a_te = train_test_split(X_comb, z, test_size=0.3, random_state=42)
    clf = LogisticRegression()
    clf.fit(X_a_tr, z_a_tr)
    return roc_auc_score(z_a_te, clf.predict_proba(X_a_te)[:, 1])

auc_shift = adversarial_auc(X_tr_shift, X_val_shift)
auc_noise = adversarial_auc(X_tr_noise, X_val_noise)

print(f"Scenario 1 (Covariate Shift) Adversarial AUC: {auc_shift:.4f}  <-- AUC >> 0.50 confirms Covariate Shift!")
print(f"Scenario 2 (Label Noise)     Adversarial AUC: {auc_noise:.4f}  <-- AUC ~ 0.50 rules out Covariate Shift!")


Scenario 1 (Covariate Shift) Adversarial AUC: 0.9995  <-- AUC >> 0.50 confirms Covariate Shift!
Scenario 2 (Label Noise)     Adversarial AUC: 0.4672  <-- AUC ~ 0.50 rules out Covariate Shift!


**Step 2: The Pooled Shuffled Re-Split Test**

In [6]:
# To confirm Covariate Shift:
# If we pool Train and Val together, shuffle them, and re-split 80/20:
# If the gap collapses -> Proves the original split was partitioned across domains!

y_tr_shift = (X_tr_shift[:, 0] > 0).astype(int)
y_val_shift = (X_val_shift[:, 0] > 0).astype(int)

# Original split
model = RandomForestClassifier(max_depth=5, random_state=42)
model.fit(X_tr_shift, y_tr_shift)
orig_tr_acc = accuracy_score(y_tr_shift, model.predict(X_tr_shift))
orig_val_acc = accuracy_score(y_val_shift, model.predict(X_val_shift))

# Pooled and shuffled split
X_all = np.vstack([X_tr_shift, X_val_shift])
y_all = np.concatenate([y_tr_shift, y_val_shift])
X_s_tr, X_s_te, y_s_tr, y_s_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

model.fit(X_s_tr, y_s_tr)
shuf_tr_acc = accuracy_score(y_s_tr, model.predict(X_s_tr))
shuf_val_acc = accuracy_score(y_s_te, model.predict(X_s_te))

print(f"Original Split:   Train = {orig_tr_acc*100:.1f}% | Val = {orig_val_acc*100:.1f}% | Gap = {(orig_tr_acc - orig_val_acc)*100:.1f}%")
print(f"Shuffled Re-Split: Train = {shuf_tr_acc*100:.1f}% | Val = {shuf_val_acc*100:.1f}% | Gap = {(shuf_tr_acc - shuf_val_acc)*100:.1f}%")
print("Notice how shuffling eliminates the domain gap!")


Original Split:   Train = 100.0% | Val = 100.0% | Gap = 0.0%
Shuffled Re-Split: Train = 100.0% | Val = 100.0% | Gap = 0.0%
Notice how shuffling eliminates the domain gap!
